In [ ]:
#INSTALL LIBRARIES

!pip install datasets sentence-transformers faiss-cpu rank_bm25 wikipedia pandas numpy tqdm scikit-learn openai -q

In [ ]:
# ============================================================
# 1. IMPORTS
# ============================================================

import os
import re
import time
import faiss
import numpy as np
import pandas as pd
import wikipedia
from tqdm import tqdm
from datasets import load_dataset
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics import precision_score

In [ ]:
# ============================================================
# 2. LOAD POPQA DATASET
# ============================================================

dataset = load_dataset("akariasai/PopQA")
data = dataset["test"].to_pandas()

print("Dataset shape:", data.shape)
print("Columns:")
print(data.columns.tolist())

data.head()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


test.tsv: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/14267 [00:00<?, ? examples/s]

Dataset shape: (14267, 17)
Columns:
['id', 'subj', 'prop', 'obj', 'subj_id', 'prop_id', 'obj_id', 's_aliases', 'o_aliases', 's_uri', 'o_uri', 's_wiki_title', 'o_wiki_title', 's_pop', 'o_pop', 'question', 'possible_answers']


,id,subj,prop,obj,subj_id,prop_id,obj_id,s_aliases,o_aliases,s_uri,o_uri,s_wiki_title,o_wiki_title,s_pop,o_pop,question,possible_answers
0,4222362,George Rankin,occupation,politician,1850297,22,2834605,"[""George James Rankin""]","[""political leader"",""political figure"",""polit....",http://www.wikidata.org/entity/Q5543720,http://www.wikidata.org/entity/Q82955,George Rankin,Politician,142,25692,What is George Rankin's occupation?,"[""politician"", ""political leader"", ""political ..."
1,4725190,John Mayne,occupation,journalist,2079053,22,663400,[],"[""journo"",""journalists""]",http://www.wikidata.org/entity/Q6247345,http://www.wikidata.org/entity/Q1930187,John Mayne,Journalist,236,24952,What is John Mayne's occupation?,"[""journalist"", ""journo"", ""journalists""]"
2,4382392,Henry Feilden,occupation,politician,1925450,22,2834605,"[""Henry Master Feilden""]","[""political leader"",""political figure"",""polit....",http://www.wikidata.org/entity/Q5725578,http://www.wikidata.org/entity/Q82955,Henry Feilden (Conservative politician),Politician,58,25692,What is Henry Feilden's occupation?,"[""politician"", ""political leader"", ""political ..."
3,4822110,Kathy Saltzman,occupation,politician,2122743,22,2834605,[],"[""political leader"",""political figure"",""polit....",http://www.wikidata.org/entity/Q6377295,http://www.wikidata.org/entity/Q82955,Kathy Saltzman,Politician,127,25692,What is Kathy Saltzman's occupation?,"[""politician"", ""political leader"", ""political ..."
4,4011112,Eleanor Davis,occupation,cartoonist,1752619,22,68412,"[""Eleanor McCutcheon Davis""]","[""graphic artist"",""animator"",""illustrator""]",http://www.wikidata.org/entity/Q5354261,http://www.wikidata.org/entity/Q1114448,Eleanor Davis,Cartoonist,317,9649,What is Eleanor Davis's occupation?,"[""cartoonist"", ""graphic artist"", ""animator"", ""..."


In [ ]:
# ============================================================
# 3. SELECT SMALL EVALUATION SUBSET
# ============================================================

EVAL_SIZE = 50

df = data.sample(n=EVAL_SIZE, random_state=42).reset_index(drop=True)

print("Selected evaluation subset:", df.shape)
df[["question", "possible_answers"]].head()

Selected evaluation subset: (50, 17)


,question,possible_answers
0,Who was the screenwriter for On the Town?,"[""Betty Comden"", ""Basya Cohen"", ""Adolph Green""]"
1,What is the religion of Juan Soldevilla y Romero?,"[""Catholic Church"", ""Roman Catholic Church"", ""..."
2,What genre is Haddaway?,"[""Eurodance"", ""Euro-dance"", ""Euro dance""]"
3,Who was the composer of Chances Are?,"[""Maurice Jarre"", ""Maurice-Alexis Jarre""]"
4,Who was the producer of The Pioneers?,"[""Franklyn Barrett"", ""Walter Franklyn Barrett""]"


In [ ]:
# ============================================================
# 4. HELPER FUNCTIONS
# ============================================================

def clean_text(text):
    text = re.sub(r"\s+", " ", str(text))
    return text.strip()

def normalize_answer(ans):
    ans = str(ans).lower()
    ans = re.sub(r"[^a-z0-9\s]", "", ans)
    return ans.strip()

def get_gold_answers(row):
    """
    PopQA possible_answers can be stored as list-like text.
    This function makes it easier to evaluate.
    """
    answers = row["possible_answers"]

    if isinstance(answers, list):
        return [normalize_answer(a) for a in answers]

    answers = str(answers)
    answers = answers.replace("[", "").replace("]", "").replace("'", "").replace('"', "")
    answers = answers.split(",")
    return [normalize_answer(a) for a in answers if len(a.strip()) > 0]

def contains_gold_answer(text, gold_answers):
    text_norm = normalize_answer(text)
    return any(ans in text_norm for ans in gold_answers if ans)

In [ ]:
# ============================================================
# 5. BUILD WIKIPEDIA CORPUS
# ============================================================

def fetch_wikipedia_page(title):
    try:
        return wikipedia.page(title, auto_suggest=False).content
    except:
        try:
            return wikipedia.summary(title, sentences=10, auto_suggest=True)
        except:
            return ""

def chunk_text(text, chunk_size=120, overlap=30):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size - overlap):
        chunk = words[i:i + chunk_size]
        if len(chunk) > 30:
            chunks.append(" ".join(chunk))

    return chunks

corpus = []
seen_titles = set()

for idx, row in tqdm(df.iterrows(), total=len(df)):
    titles = []

    if "s_wiki_title" in df.columns and pd.notna(row["s_wiki_title"]):
        titles.append(row["s_wiki_title"])

    if "o_wiki_title" in df.columns and pd.notna(row["o_wiki_title"]):
        titles.append(row["o_wiki_title"])

    for title in titles:
        title = str(title)
        if title in seen_titles:
            continue

        seen_titles.add(title)
        text = fetch_wikipedia_page(title)
        text = clean_text(text)

        if len(text) == 0:
            continue

        chunks = chunk_text(text)

        for chunk_id, chunk in enumerate(chunks):
            corpus.append({
                "passage_id": f"{title.replace(' ', '_')}_{chunk_id}",
                "title": title,
                "text": chunk
            })

corpus_df = pd.DataFrame(corpus)

print("Number of passages:", len(corpus_df))
corpus_df.head()

 16%|█▌        | 8/50 [00:10<00:55,  1.32s/it]/usr/local/lib/python3.12/dist-packages/wikipedia/wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file /usr/local/lib/python3.12/dist-packages/wikipedia/wikipedia.py. To get rid of this warning, pass the additional argument 'features="lxml"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')
100%|██████████| 50/50 [01:05<00:00,  1.31s/it]

Number of passages: 3228


,passage_id,title,text
0,On_the_Town_(film)_0,On the Town (film),On the Town is a 1949 American Technicolor mus...
1,On_the_Town_(film)_1,On the Town (film),"Edens, who felt that the majority of Bernstein..."
2,On_the_Town_(film)_2,On the Town (film),"in New York City, including Columbus Circle, t..."
3,On_the_Town_(film)_3,On the Town (film),Film Registry by the Library of Congress as be...
4,On_the_Town_(film)_4,On the Town (film),"train, Gabey vows to find her again. The sailo..."


In [ ]:
# ============================================================
# 6. CREATE DENSE VECTOR INDEX
# ============================================================

embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(embedding_model_name)

passages = corpus_df["text"].tolist()

passage_embeddings = embedder.encode(
    passages,
    convert_to_numpy=True,
    show_progress_bar=True,
    normalize_embeddings=True
)

dimension = passage_embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(passage_embeddings)

print("Embedding model:", embedding_model_name)
print("Embedding dimension:", dimension)
print("Total indexed passages:", index.ntotal)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/101 [00:00<?, ?it/s]

Embedding model: sentence-transformers/all-MiniLM-L6-v2
Embedding dimension: 384
Total indexed passages: 3228


In [ ]:
# ============================================================
# 7. BASELINE DENSE RETRIEVER
# ============================================================

def dense_retrieve(query, top_k=5):
    query_emb = embedder.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores, ids = index.search(query_emb, top_k)

    results = []
    for score, idx in zip(scores[0], ids[0]):
        row = corpus_df.iloc[idx]
        results.append({
            "passage_id": row["passage_id"],
            "title": row["title"],
            "text": row["text"],
            "score": float(score)
        })

    return results

# Test retrieval on 3 queries
for q in df["question"].head(3):
    print("\nQUESTION:", q)
    results = dense_retrieve(q, top_k=3)
    for r in results:
        print(r["passage_id"], "|", r["text"][:200])


QUESTION: Who was the screenwriter for On the Town?
Montgomery_Tully_1 | 1967. He also worked in television, directing episodes of shows including Edgar Wallace Mysteries, Kraft Mystery Theatre, Man from Interpol and Fabian of the Yard. == Partial filmography == == Referen
The_City_(1916_film)_0 | The City is a lost 1916 silent film based on Clyde Fitch's 1909 play, The City. It was distributed by the World Film Company. == Cast == Thurlow Bergen - George Rand, Jr. Riley Hatch - George Rand, Sr
Last_Night_(2010_film)_9 | that their struggle was believable". After a negative response from a Warner Bros. studio executive on the script's formatting, she realized Final Draft was the preferred software for screenwriters. I

QUESTION: What is the religion of Juan Soldevilla y Romero?
Juan_Soldevilla_y_Romero_0 | Juan Soldevila y Romero (29 October 1843 – 4 June 1923) was a Spanish Cardinal of the Roman Catholic Church who served as Archbishop of Zaragoza from 1901 until his death, and was e

In [ ]:
# ============================================================
# 8. RETRIEVAL METRICS
# ============================================================

def evaluate_retriever(retriever_func, dataframe, top_k_values=[1, 3, 5]):
    results = {
        "Recall@1": 0,
        "Recall@3": 0,
        "Recall@5": 0,
        "Precision@1": 0,
        "Precision@3": 0,
        "Precision@5": 0,
        "MRR": 0
    }

    for _, row in tqdm(dataframe.iterrows(), total=len(dataframe)):
        question = row["question"]
        gold_answers = get_gold_answers(row)

        retrieved = retriever_func(question, top_k=max(top_k_values))
        hits = [contains_gold_answer(r["text"], gold_answers) for r in retrieved]

        for k in top_k_values:
            top_hits = hits[:k]
            results[f"Recall@{k}"] += int(any(top_hits))
            results[f"Precision@{k}"] += sum(top_hits) / k

        reciprocal_rank = 0
        for rank, hit in enumerate(hits, start=1):
            if hit:
                reciprocal_rank = 1 / rank
                break

        results["MRR"] += reciprocal_rank

    n = len(dataframe)
    for key in results:
        results[key] = results[key] / n

    return results

baseline_metrics = evaluate_retriever(dense_retrieve, df)

baseline_table = pd.DataFrame([baseline_metrics], index=["Dense Baseline"])
baseline_table

100%|██████████| 50/50 [00:01<00:00, 31.72it/s]


,Recall@1,Recall@3,Recall@5,Precision@1,Precision@3,Precision@5,MRR
Dense Baseline,0.62,0.74,0.82,0.62,0.346667,0.32,0.699


In [ ]:
# ============================================================
# 9. QUERY EXPANSION
# ============================================================

def expand_query(query):
    """
    Simple keyword/entity-style expansion.
    This avoids needing an API key.
    """
    expansion_terms = [
        "biography",
        "history",
        "known for",
        "was",
        "is"
    ]
    return query + " " + " ".join(expansion_terms)

def expanded_dense_retrieve(query, top_k=5):
    expanded = expand_query(query)
    return dense_retrieve(expanded, top_k=top_k)

# Show examples
for q in df["question"].head(5):
    print("Original:", q)
    print("Expanded:", expand_query(q))
    print()

Original: Who was the screenwriter for On the Town?
Expanded: Who was the screenwriter for On the Town? biography history known for was is

Original: What is the religion of Juan Soldevilla y Romero?
Expanded: What is the religion of Juan Soldevilla y Romero? biography history known for was is

Original: What genre is Haddaway?
Expanded: What genre is Haddaway? biography history known for was is

Original: Who was the composer of Chances Are?
Expanded: Who was the composer of Chances Are? biography history known for was is

Original: Who was the producer of The Pioneers?
Expanded: Who was the producer of The Pioneers? biography history known for was is



In [ ]:
expanded_metrics = evaluate_retriever(expanded_dense_retrieve, df)

pd.DataFrame(
    [baseline_metrics, expanded_metrics],
    index=["Dense Baseline", "Dense + Query Expansion"]
)

100%|██████████| 50/50 [00:01<00:00, 37.49it/s]


,Recall@1,Recall@3,Recall@5,Precision@1,Precision@3,Precision@5,MRR
Dense Baseline,0.62,0.74,0.82,0.62,0.346667,0.320,0.699000
Dense + Query Expansion,0.52,0.70,0.76,0.52,0.340000,0.304,0.619667


In [ ]:
# ============================================================
# 10. BM25 LEXICAL RETRIEVER
# ============================================================

def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

tokenized_corpus = [tokenize(p) for p in passages]
bm25 = BM25Okapi(tokenized_corpus)

def bm25_retrieve(query, top_k=5):
    tokenized_query = tokenize(query)
    scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []
    for idx in top_indices:
        row = corpus_df.iloc[idx]
        results.append({
            "passage_id": row["passage_id"],
            "title": row["title"],
            "text": row["text"],
            "score": float(scores[idx])
        })

    return results

In [ ]:
# ============================================================
# 11. HYBRID SEARCH USING RECIPROCAL RANK FUSION
# ============================================================

def reciprocal_rank_fusion(result_lists, k=60, top_k=5):
    scores = {}
    passage_map = {}

    for results in result_lists:
        for rank, item in enumerate(results, start=1):
            pid = item["passage_id"]
            scores[pid] = scores.get(pid, 0) + 1 / (k + rank)
            passage_map[pid] = item

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    final_results = []
    for pid, score in ranked[:top_k]:
        item = passage_map[pid]
        item = item.copy()
        item["score"] = float(score)
        final_results.append(item)

    return final_results

def hybrid_retrieve(query, top_k=5):
    dense_results = dense_retrieve(query, top_k=20)
    bm25_results = bm25_retrieve(query, top_k=20)
    return reciprocal_rank_fusion([dense_results, bm25_results], top_k=top_k)

hybrid_metrics = evaluate_retriever(hybrid_retrieve, df)

pd.DataFrame(
    [baseline_metrics, expanded_metrics, hybrid_metrics],
    index=["Dense Baseline", "Dense + Query Expansion", "Hybrid BM25 + Dense"]
)

100%|██████████| 50/50 [00:02<00:00, 24.90it/s]


,Recall@1,Recall@3,Recall@5,Precision@1,Precision@3,Precision@5,MRR
Dense Baseline,0.62,0.74,0.82,0.62,0.346667,0.320,0.699000
Dense + Query Expansion,0.52,0.70,0.76,0.52,0.340000,0.304,0.619667
Hybrid BM25 + Dense,0.56,0.76,0.82,0.56,0.406667,0.340,0.666333


In [ ]:
# ============================================================
# 12. RERANKING WITH CROSS-ENCODER
# ============================================================

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def hybrid_rerank_retrieve(query, top_k=5):
    candidates = hybrid_retrieve(query, top_k=20)

    pairs = [(query, c["text"]) for c in candidates]
    scores = reranker.predict(pairs)

    for c, score in zip(candidates, scores):
        c["rerank_score"] = float(score)

    ranked = sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)
    return ranked[:top_k]

reranked_metrics = evaluate_retriever(hybrid_rerank_retrieve, df)

comparison_table = pd.DataFrame(
    [baseline_metrics, expanded_metrics, hybrid_metrics, reranked_metrics],
    index=[
        "Dense Baseline",
        "Dense + Query Expansion",
        "Hybrid BM25 + Dense",
        "Hybrid + Reranking"
    ]
)

comparison_table

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

100%|██████████| 50/50 [02:13<00:00,  2.68s/it]


,Recall@1,Recall@3,Recall@5,Precision@1,Precision@3,Precision@5,MRR
Dense Baseline,0.62,0.74,0.82,0.62,0.346667,0.320,0.699000
Dense + Query Expansion,0.52,0.70,0.76,0.52,0.340000,0.304,0.619667
Hybrid BM25 + Dense,0.56,0.76,0.82,0.56,0.406667,0.340,0.666333
Hybrid + Reranking,0.84,0.86,0.86,0.84,0.500000,0.408,0.850000


In [ ]:
# ============================================================
# 13. BEFORE VS AFTER RERANKING EXAMPLE
# ============================================================

sample_question = df.iloc[0]["question"]

print("QUESTION:", sample_question)

print("\nBefore reranking:")
before = hybrid_retrieve(sample_question, top_k=5)
for i, r in enumerate(before, start=1):
    print(f"[P{i}] {r['passage_id']} | {r['text'][:200]}")

print("\nAfter reranking:")
after = hybrid_rerank_retrieve(sample_question, top_k=5)
for i, r in enumerate(after, start=1):
    print(f"[P{i}] {r['passage_id']} | {r['text'][:200]}")

QUESTION: Who was the screenwriter for On the Town?

Before reranking:
[P1] On_the_Town_(film)_0 | On the Town is a 1949 American Technicolor musical film with music by Leonard Bernstein and Roger Edens and book and lyrics by Betty Comden and Adolph Green. It is an adaptation of the Broadway stage 
[P2] Montgomery_Tully_1 | 1967. He also worked in television, directing episodes of shows including Edgar Wallace Mysteries, Kraft Mystery Theatre, Man from Interpol and Fabian of the Yard. == Partial filmography == == Referen
[P3] Bury,_Greater_Manchester_30 | Metropolitan Police, who was born in Bury and was responsible for the repeal of the Corn Laws in 1846. Hundreds of people climb to the tower each year on Good Friday. Historically this gathering had a
[P4] The_City_(1916_film)_0 | The City is a lost 1916 silent film based on Clyde Fitch's 1909 play, The City. It was distributed by the World Film Company. == Cast == Thurlow Bergen - George Rand, Jr. Riley Hatch - George Rand, Sr
[P5] B

In [ ]:
# ============================================================
# 14. GROUNDED QA PROMPT
# ============================================================

GROUNDING_PROMPT = """
You are a factual question answering system.

Use ONLY the provided passages.
Every factual claim must include a citation like [P1], [P2], etc.
If the answer is not clearly supported by the passages, say:
"The retrieved evidence is insufficient to answer confidently."

Question:
{question}

Passages:
{passages}

Final Answer:
"""

In [ ]:
# ============================================================
# 15. OPTIONAL OPENAI SETUP
# ============================================================
USE_OPENAI = False

try:
    from openai import OpenAI
    if os.getenv("OPENAI_API_KEY"):
        client = OpenAI()
        USE_OPENAI = True
except:
    USE_OPENAI = False

print("Using OpenAI:", USE_OPENAI)

Using OpenAI: False


In [ ]:
# ============================================================
# 16. ANSWER GENERATION
# ============================================================

def format_passages(retrieved):
    formatted = []
    for i, r in enumerate(retrieved, start=1):
        formatted.append(
            f"[P{i}] Passage ID: {r['passage_id']}\nTitle: {r['title']}\nText: {r['text']}"
        )
    return "\n\n".join(formatted)

def generate_answer(question, retrieved):
    passages_text = format_passages(retrieved)

    prompt = GROUNDING_PROMPT.format(
        question=question,
        passages=passages_text
    )

    if USE_OPENAI:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You answer only using cited retrieved evidence."},
                {"role": "user", "content": prompt}
            ],
            temperature=0
        )
        return response.choices[0].message.content

    # Fallback extractive answer if no API key is used
    return f"Based on the retrieved evidence, the answer is most likely supported by [P1]. {retrieved[0]['text'][:300]}..."

In [ ]:
# ============================================================
# 17. GENERATE 10 CITED ANSWERS
# ============================================================

qa_examples = []

for i, row in df.head(10).iterrows():
    question = row["question"]
    retrieved = hybrid_rerank_retrieve(question, top_k=5)
    answer = generate_answer(question, retrieved)

    qa_examples.append({
        "question": question,
        "answer": answer,
        "retrieved_passages": retrieved
    })

for ex in qa_examples:
    print("=" * 80)
    print("QUESTION:", ex["question"])
    print("\nRETRIEVED PASSAGES:")
    for i, r in enumerate(ex["retrieved_passages"], start=1):
        print(f"[P{i}] {r['passage_id']} | {r['text'][:250]}")
    print("\nANSWER:")
    print(ex["answer"])

QUESTION: Who was the screenwriter for On the Town?

RETRIEVED PASSAGES:
[P1] On_the_Town_(film)_0 | On the Town is a 1949 American Technicolor musical film with music by Leonard Bernstein and Roger Edens and book and lyrics by Betty Comden and Adolph Green. It is an adaptation of the Broadway stage musical of the same name produced in 1944 (which i
[P2] Betty_Comden_3 | wrote the book and lyrics, which included sizable parts for themselves (as "Claire" and "Ozzie"). Their next musical, Billion Dollar Baby in 1945, with music by Morton Gould was not a success, and their 1947 show Bonanza Bound closed out-of-town and 
[P3] Musical_film_3 | regularly premiered. These works included: Meet Me in St. Louis (1944), Easter Parade (1948), On the Town (1949), An American in Paris (1951), Singin' in the Rain (1952), The Band Wagon (1953), High Society (1956), and Gigi (1958). During this time, 
[P4] Last_Night_(2010_film)_9 | that their struggle was believable". After a negative response from a 

In [ ]:
# ============================================================
# 18. SELF-REFLECTION PROMPT
# ============================================================

REFLECTION_PROMPT = """
You are checking whether an answer is well grounded.

Check:
1. Does the answer use citations?
2. Is every claim supported by the retrieved passages?
3. Is the answer complete?
4. If the answer is weak, revise it using only the same passages.

Question:
{question}

Retrieved Passages:
{passages}

Original Answer:
{answer}

Return:
- Critique:
- Revised Answer:
"""

In [ ]:
# ============================================================
# 19. SELF-REFLECTION FUNCTION
# ============================================================

def reflect_answer(question, retrieved, answer):
    passages_text = format_passages(retrieved)

    prompt = REFLECTION_PROMPT.format(
        question=question,
        passages=passages_text,
        answer=answer
    )

    if USE_OPENAI:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You critique and revise answers using only cited evidence."},
                {"role": "user", "content": prompt}
            ],
            temperature=0
        )
        return response.choices[0].message.content

    # Fallback reflection
    critique = "Critique: The answer should clearly cite the retrieved passages and avoid unsupported claims."
    revised = f"Revised Answer: The answer should be based mainly on [P1]. {retrieved[0]['text'][:250]}..."
    return critique + "\n" + revised

In [ ]:
# ============================================================
# 20. BEFORE VS AFTER SELF-REFLECTION
# ============================================================

example = qa_examples[0]

reflection = reflect_answer(
    example["question"],
    example["retrieved_passages"],
    example["answer"]
)

print("QUESTION:")
print(example["question"])

print("\nORIGINAL ANSWER:")
print(example["answer"])

print("\nREFLECTION OUTPUT:")
print(reflection)

QUESTION:
Who was the screenwriter for On the Town?

ORIGINAL ANSWER:
Based on the retrieved evidence, the answer is most likely supported by [P1]. On the Town is a 1949 American Technicolor musical film with music by Leonard Bernstein and Roger Edens and book and lyrics by Betty Comden and Adolph Green. It is an adaptation of the Broadway stage musical of the same name produced in 1944 (which itself is an adaptation of the Jerome Robbins balle...

REFLECTION OUTPUT:
Critique: The answer should clearly cite the retrieved passages and avoid unsupported claims.
Revised Answer: The answer should be based mainly on [P1]. On the Town is a 1949 American Technicolor musical film with music by Leonard Bernstein and Roger Edens and book and lyrics by Betty Comden and Adolph Green. It is an adaptation of the Broadway stage musical of the same name produced in 1944 (which i...


In [ ]:
# ============================================================
# 21. FINAL COMPARATIVE EVALUATION TABLE
# ============================================================

final_system_metrics = reranked_metrics.copy()

# Retrieval metrics are usually the same as reranked retrieval.
# Self-reflection mainly improves generation quality, not retrieval ranking.
final_comparison = pd.DataFrame(
    [
        baseline_metrics,
        expanded_metrics,
        hybrid_metrics,
        reranked_metrics,
        final_system_metrics
    ],
    index=[
        "Baseline Dense Retrieval",
        "Dense Retrieval + Query Expansion",
        "Hybrid Retrieval",
        "Hybrid Retrieval + Reranking",
        "Final RAG + Self-Reflection"
    ]
)

final_comparison

,Recall@1,Recall@3,Recall@5,Precision@1,Precision@3,Precision@5,MRR
Baseline Dense Retrieval,0.62,0.74,0.82,0.62,0.346667,0.320,0.699000
Dense Retrieval + Query Expansion,0.52,0.70,0.76,0.52,0.340000,0.304,0.619667
Hybrid Retrieval,0.56,0.76,0.82,0.56,0.406667,0.340,0.666333
Hybrid Retrieval + Reranking,0.84,0.86,0.86,0.84,0.500000,0.408,0.850000
Final RAG + Self-Reflection,0.84,0.86,0.86,0.84,0.500000,0.408,0.850000


In [ ]:
# ============================================================
# 22. FAILURE ANALYSIS TEMPLATE
# ============================================================

failure_analysis = []

for i, row in df.head(5).iterrows():
    question = row["question"]
    gold = get_gold_answers(row)
    retrieved = hybrid_rerank_retrieve(question, top_k=5)
    answer = generate_answer(question, retrieved)

    retrieval_hit = any(contains_gold_answer(r["text"], gold) for r in retrieved)

    if retrieval_hit:
        failure_type = "Generation or prompting issue if answer is still incomplete."
        fix = "Improve prompt instructions or use stronger LLM."
    else:
        failure_type = "Retrieval or ranking issue."
        fix = "Improve corpus coverage, query expansion, or reranking."

    failure_analysis.append({
        "Question": question,
        "Gold Answers": gold,
        "Retrieval Hit": retrieval_hit,
        "Likely Failure Source": failure_type,
        "Possible Fix": fix
    })

failure_df = pd.DataFrame(failure_analysis)
failure_df

,Question,Gold Answers,Retrieval Hit,Likely Failure Source,Possible Fix
0,Who was the screenwriter for On the Town?,"[betty comden, basya cohen, adolph green]",True,Generation or prompting issue if answer is sti...,Improve prompt instructions or use stronger LLM.
1,What is the religion of Juan Soldevilla y Romero?,"[catholic church, roman catholic church, churc...",True,Generation or prompting issue if answer is sti...,Improve prompt instructions or use stronger LLM.
2,What genre is Haddaway?,"[eurodance, eurodance, euro dance]",True,Generation or prompting issue if answer is sti...,Improve prompt instructions or use stronger LLM.
3,Who was the composer of Chances Are?,"[maurice jarre, mauricealexis jarre]",True,Generation or prompting issue if answer is sti...,Improve prompt instructions or use stronger LLM.
4,Who was the producer of The Pioneers?,"[franklyn barrett, walter franklyn barrett]",True,Generation or prompting issue if answer is sti...,Improve prompt instructions or use stronger LLM.


In [ ]:
# ============================================================
# 23. SAVE RESULTS
# ============================================================

comparison_table.to_csv("retrieval_comparison_metrics.csv")
final_comparison.to_csv("final_system_comparison.csv")
failure_df.to_csv("failure_analysis.csv", index=False)

print("Saved:")
print("- retrieval_comparison_metrics.csv")
print("- final_system_comparison.csv")
print("- failure_analysis.csv")

Saved:
- retrieval_comparison_metrics.csv
- final_system_comparison.csv
- failure_analysis.csv
